# SPK Seleksi Penerima Beasiswa KIP-K dengan PROMETHEE II

Notebook ini mengimplementasikan pipeline SPK KIP-K Politeknik Negeri Jember:

`Data Mentah -> Preprocessing -> Penentuan Kriteria -> Pembobotan -> Perhitungan PROMETHEE -> Ranking -> Validasi -> Export Model`

Jalankan berurutan dari atas ke bawah menggunakan `Kernel -> Restart & Run All`.

In [ ]:
# --- TAHAP 0 - INSTALL LIBRARY DAN KONFIGURASI -----------------------------

import json
import os
import re
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# Path data
DATA_PATH = "data/raw/V_Data_Mahasiswa_Full_KIPK.xlsx"
DITETAPKAN_PATH = "data/raw/Mahasiswa_ditetapkan_Politeknik_Negeri_Jember_20241012.xlsx"
DITETAPKAN_PATH_FALLBACKS = [
    "data/raw/Mahasiswa_ditetapkan_Politeknik Negeri Jember_20241012.xlsx",
]
CLEAN_PATH = "data/processed/data_clean.csv"

# Path output
OUT_RANKING = "output/hasil_ranking.xlsx"
OUT_VALIDASI = "output/validasi_result.json"
OUT_MODEL = "output/model/model_promethee.joblib"
OUT_CONFIG = "output/model/scaler_config.json"
OUT_FIG_DIST = "output/figures/distribusi_kriteria.png"
OUT_FIG_FLOW = "output/figures/promethee_flow.png"
OUT_FIG_CM = "output/figures/confusion_matrix.png"

# Kuota penerima
KUOTA = 50

# Konfigurasi kriteria dan bobot
CRITERIA = {
    "C1": {"name": "Kepemilikan KIP SMA", "weight": 0.3, "type": "benefit"},
    "C2": {"name": "Status DTKS", "weight": 0.2, "type": "benefit"},
    "C3": {"name": "Desil", "weight": 0.2, "type": "benefit"},
    "C4": {"name": "Penghasilan Orang Tua", "weight": 0.1, "type": "benefit"},
    "C5": {"name": "Status Orang Tua", "weight": 0.1, "type": "benefit"},
    "C6": {"name": "Prestasi", "weight": 0.1, "type": "benefit"},
}

# Mapping sub-kriteria teks ke nilai numerik
SUB_CRITERIA_MAP = {
    "C1": {
        "Memiliki KIP": 4,
        "Tidak memiliki KIP, tetapi termasuk keluarga penerima bantuan sosial lain": 3,
        "Tidak memiliki KIP, tetapi memiliki SKTM dari pihak berwenang": 2,
        "Tidak memiliki semuanya": 1,
    },
    "C2": {
        "Terdaftar dalam DTKS": 3,
        "Penerima bantuan sosial": 2,
        "Belum terdaftar": 1,
    },
    "C3": {
        "Desil 1": 4,
        "Desil 2": 3,
        "Desil 3": 2,
        "Desil > 4": 1,
    },
    "C4": {
        "<= Rp 1.000.000": 5,
        "> Rp 1.000.000 s.d. <= Rp 2.000.000": 4,
        "> Rp 2.000.000 s.d. <= Rp 3.000.000": 3,
        "> Rp 3.000.000 s.d. <= Rp 4.000.000": 2,
        "> Rp 4.000.000": 1,
    },
    "C5": {
        "Yatim Piatu": 4,
        "Yatim / Piatu": 3,
        "Orang tua sakit / tidak bekerja": 2,
        "Orang tua masih bekerja": 1,
    },
    "C6": {
        "Internasional": 5,
        "Nasional": 4,
        "Provinsi / Kota": 3,
        "Sekolah": 2,
        "Tidak ada": 1,
    },
}

for folder in ["data/processed", "output/model", "output/figures"]:
    os.makedirs(folder, exist_ok=True)

def normalize_text(value):
    if pd.isna(value):
        return ""
    text = str(value).strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text

def normalize_column_name(value):
    text = normalize_text(value)
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")

def resolve_existing_path(path, fallbacks=None):
    candidates = [path] + list(fallbacks or [])
    for candidate in candidates:
        if Path(candidate).exists():
            return candidate
    raise FileNotFoundError("File tidak ditemukan. Kandidat path: " + ", ".join(candidates))

def find_column(df, candidates=None, contains_any=None, contains_all=None, exclude_any=None):
    candidates = [normalize_column_name(c) for c in (candidates or [])]
    contains_any = [normalize_column_name(c) for c in (contains_any or [])]
    contains_all = [normalize_column_name(c) for c in (contains_all or [])]
    exclude_any = [normalize_column_name(c) for c in (exclude_any or [])]
    normalized = {col: normalize_column_name(col) for col in df.columns}

    for col, norm in normalized.items():
        if norm in candidates:
            return col
    for col, norm in normalized.items():
        if exclude_any and any(token in norm for token in exclude_any):
            continue
        if contains_all and not all(token in norm for token in contains_all):
            continue
        if contains_any and not any(token in norm for token in contains_any):
            continue
        if contains_any or contains_all:
            return col
    return None

def clean_identifier(series):
    return series.astype(str).str.strip().str.replace(r"\.0$", "", regex=True).str.upper()

def read_label_excel(path):
    df_candidate = pd.read_excel(path)
    if find_column(df_candidate, candidates=["NIM"]):
        return df_candidate
    return pd.read_excel(path, header=1)

def parse_income_to_amount(value):
    text = normalize_text(value)
    if text in {"", "-", "_", "nan", "none", "tidak berpenghasilan"}:
        return 0.0
    numbers = re.findall(r"\d[\d\.]*", text)
    if not numbers:
        return np.nan
    parsed = []
    for number in numbers:
        digits = re.sub(r"[^0-9]", "", number)
        if digits:
            parsed.append(float(digits))
    if not parsed:
        return np.nan
    return max(parsed)

def income_to_score(amount):
    if pd.isna(amount):
        return 1
    if amount <= 1_000_000:
        return 5
    if amount <= 2_000_000:
        return 4
    if amount <= 3_000_000:
        return 3
    if amount <= 4_000_000:
        return 2
    return 1

print("Konfigurasi selesai")

In [ ]:
# --- TAHAP 1 - LOAD DATA MENTAH --------------------------------------------

try:
    data_path = resolve_existing_path(DATA_PATH)
    ditetapkan_path = resolve_existing_path(DITETAPKAN_PATH, DITETAPKAN_PATH_FALLBACKS)
    df_raw = pd.read_excel(data_path)
    df_label = read_label_excel(ditetapkan_path)
except FileNotFoundError as error:
    print(f"ERROR: {error}")
    raise

print(f"Data pendaftar dibaca dari: {data_path}")
print(f"Shape df_raw: {df_raw.shape}")
print("Kolom df_raw:")
print(list(df_raw.columns))
display(df_raw.head())

print(f"Data ground truth dibaca dari: {ditetapkan_path}")
print(f"Shape df_label: {df_label.shape}")
print("Kolom df_label:")
print(list(df_label.columns))
display(df_label.head())

print("Load data mentah selesai")

In [ ]:
# --- TAHAP 2 - PREPROCESSING ------------------------------------------------

def is_empty_like(value):
    text = normalize_text(value)
    return text in {"", "-", "_", "nan", "none", "null"}

def fill_missing_values(df):
    filled = df.copy()
    for col in filled.columns:
        if pd.api.types.is_numeric_dtype(filled[col]):
            median = filled[col].median()
            filled[col] = filled[col].fillna(0 if pd.isna(median) else median)
        else:
            values = filled[col].replace(r"^\s*$", np.nan, regex=True)
            mode = values.dropna().mode()
            replacement = mode.iloc[0] if not mode.empty else "Tidak ada"
            filled[col] = values.fillna(replacement)
    return filled

def score_c1(row, kip_col, fallback_cols):
    texts = []
    if kip_col and kip_col in row.index:
        texts.append(normalize_text(row[kip_col]))
    for col in fallback_cols:
        if col and col in row.index:
            texts.append(normalize_text(row[col]))
    text = " ".join(texts)
    if any(token in text for token in ["kip", "kartu indonesia pintar"]):
        return 4
    if any(token in text for token in ["bansos", "bantuan sosial", "kks", "pkh", "beasiswa"]):
        return 3
    if "sktm" in text:
        return 2
    return 1

def score_c2(value):
    text = normalize_text(value)
    if any(token in text for token in ["terdaftar", "terdata", "dtks", "ya"]):
        return 3
    if any(token in text for token in ["bansos", "bantuan", "kks", "pkh"]):
        return 2
    return 1

def score_c3(value):
    text = normalize_text(value)
    match = re.search(r"(\d+)", text)
    if not match:
        return 1
    desil = int(match.group(1))
    if desil == 1:
        return 4
    if desil == 2:
        return 3
    if desil == 3:
        return 2
    return 1

def score_c5(row, status_ayah_col, status_ibu_col, kerja_ayah_col, kerja_ibu_col):
    status_ayah = normalize_text(row[status_ayah_col]) if status_ayah_col else ""
    status_ibu = normalize_text(row[status_ibu_col]) if status_ibu_col else ""
    kerja_ayah = normalize_text(row[kerja_ayah_col]) if kerja_ayah_col else ""
    kerja_ibu = normalize_text(row[kerja_ibu_col]) if kerja_ibu_col else ""
    ayah_meninggal = any(token in status_ayah for token in ["meninggal", "wafat"])
    ibu_meninggal = any(token in status_ibu for token in ["meninggal", "wafat"])
    if ayah_meninggal and ibu_meninggal:
        return 4
    if ayah_meninggal or ibu_meninggal:
        return 3
    combined = " ".join([status_ayah, status_ibu, kerja_ayah, kerja_ibu])
    if any(token in combined for token in ["tidak bekerja", "sakit", "disabilitas"]):
        return 2
    return 1

def score_c6(value):
    text = normalize_text(value)
    if text in {"", "-", "_", "0", "tidak ada", "belum ada", "tidak berprestasi", "(-)", "\"-\""}:
        return 1
    if "internasional" in text:
        return 5
    if "nasional" in text:
        return 4
    if any(token in text for token in ["provinsi", "kabupaten", "kota", "kejurda", "kejurb", "bupati", "walikota", "jawa timur"]):
        return 3
    return 2

nim_col = find_column(df_raw, candidates=["nim"], contains_any=["nim"])
nama_col = find_column(df_raw, candidates=["nama"], contains_any=["nama"])
kip_col = find_column(df_raw, contains_any=["kip", "no_kip", "kartu_indonesia_pintar"])
dtks_col = find_column(df_raw, candidates=["dtk", "dtks", "status_dtks"], contains_any=["dtk", "dtks"])
desil_col = find_column(df_raw, candidates=["desil"], contains_any=["desil", "p3ke"])
penghasilan_ayah_col = find_column(df_raw, candidates=["penghasilan_ayah"], contains_all=["penghasilan", "ayah"])
penghasilan_ibu_col = find_column(df_raw, candidates=["penghasilan_ibu"], contains_all=["penghasilan", "ibu"])
status_ayah_col = find_column(df_raw, candidates=["keterangan_ayah", "status_ayah"], contains_any=["keterangan_ayah", "status_ayah"])
status_ibu_col = find_column(df_raw, candidates=["keterangan_ibu", "status_ibu"], contains_any=["keterangan_ibu", "status_ibu"])
kerja_ayah_col = find_column(df_raw, candidates=["kerja_ayah", "pekerjaan_ayah"], contains_any=["kerja_ayah", "pekerjaan_ayah"])
kerja_ibu_col = find_column(df_raw, candidates=["kerja_ibu", "pekerjaan_ibu"], contains_any=["kerja_ibu", "pekerjaan_ibu"])
prestasi_col = find_column(df_raw, candidates=["prestasi_olahraga", "prestasi"], contains_any=["prestasi"])
fallback_c1_cols = [
    find_column(df_raw, candidates=["lembaga"]),
    find_column(df_raw, candidates=["sumber_biaya"]),
    find_column(df_raw, candidates=["jenis_lembaga"]),
]

required_detected = {
    "NIM": nim_col,
    "Nama": nama_col,
    "C2 Status DTKS": dtks_col,
    "C3 Desil": desil_col,
    "C4 Penghasilan Ayah": penghasilan_ayah_col,
    "C4 Penghasilan Ibu": penghasilan_ibu_col,
    "C5 Status Ayah": status_ayah_col,
    "C5 Status Ibu": status_ibu_col,
    "C6 Prestasi": prestasi_col,
}
missing_required = [name for name, col in required_detected.items() if col is None]
if missing_required:
    raise ValueError("Kolom wajib tidak terdeteksi: " + ", ".join(missing_required))

df_work = df_raw.copy()
df_work[nim_col] = clean_identifier(df_work[nim_col])
duplicate_count = int(df_work.duplicated(subset=[nim_col]).sum())
df_work = df_work.drop_duplicates(subset=[nim_col]).reset_index(drop=True)
missing_before = df_work.isna().sum()
df_filled = fill_missing_values(df_work)

ayah_amount = df_filled[penghasilan_ayah_col].apply(parse_income_to_amount)
ibu_amount = df_filled[penghasilan_ibu_col].apply(parse_income_to_amount)
total_penghasilan = ayah_amount.fillna(0) + ibu_amount.fillna(0)

df_clean = pd.DataFrame()
df_clean["NIM"] = clean_identifier(df_filled[nim_col])
df_clean["Nama"] = df_filled[nama_col].astype(str).str.strip()
program_col = find_column(df_filled, candidates=["program_studi"], contains_all=["program", "studi"])
if program_col:
    df_clean["Program_Studi"] = df_filled[program_col]
df_clean["C1"] = df_filled.apply(lambda row: score_c1(row, kip_col, fallback_c1_cols), axis=1)
df_clean["C2"] = df_filled[dtks_col].apply(score_c2)
df_clean["C3"] = df_filled[desil_col].apply(score_c3)
df_clean["C4"] = total_penghasilan.apply(income_to_score)
df_clean["C5"] = df_filled.apply(lambda row: score_c5(row, status_ayah_col, status_ibu_col, kerja_ayah_col, kerja_ibu_col), axis=1)
df_clean["C6"] = df_filled[prestasi_col].apply(score_c6)

for code in CRITERIA:
    df_clean[code] = pd.to_numeric(df_clean[code], errors="coerce").fillna(1).astype(int)

Path(CLEAN_PATH).parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(CLEAN_PATH, index=False)

print("Kolom terdeteksi:")
display(pd.DataFrame([{"Kebutuhan": key, "Kolom": value} for key, value in required_detected.items()]))
if kip_col is None:
    print("Catatan: kolom eksplisit KIP SMA tidak ditemukan. C1 dihitung dari fallback lembaga/sumber_biaya/jenis_lembaga bila tersedia, selain itu bernilai 1.")
print(f"Jumlah duplikat NIM dihapus: {duplicate_count}")
print("Jumlah missing value sebelum imputasi:")
display(missing_before[missing_before > 0].to_frame("missing_count"))
display(df_clean.head())
print(f"Data clean disimpan ke: {CLEAN_PATH}")
print("Preprocessing selesai")

In [ ]:
# --- TAHAP 3 - PENENTUAN KRITERIA DAN PEMBOBOTAN ---------------------------

criteria_codes = list(CRITERIA.keys())
total_weight = sum(item["weight"] for item in CRITERIA.values())
if not np.isclose(total_weight, 1.0):
    raise ValueError(f"Total bobot harus 1.0, saat ini {total_weight}")

criteria_table = pd.DataFrame([
    {"Kode": code, "Nama": cfg["name"], "Bobot": cfg["weight"], "Tipe": cfg["type"]}
    for code, cfg in CRITERIA.items()
])
display(criteria_table)

print("Statistik deskriptif kriteria:")
display(df_clean[criteria_codes].describe())

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()
for idx, code in enumerate(criteria_codes):
    counts = df_clean[code].value_counts().reindex(range(1, 6), fill_value=0)
    axes[idx].bar(counts.index.astype(str), counts.values, color="#2f7d57")
    axes[idx].set_title(f"{code} - {CRITERIA[code]['name']}")
    axes[idx].set_xlabel("Nilai")
    axes[idx].set_ylabel("Jumlah")
plt.tight_layout()
Path(OUT_FIG_DIST).parent.mkdir(parents=True, exist_ok=True)
plt.savefig(OUT_FIG_DIST, dpi=150, bbox_inches="tight")
plt.show()

print(f"Grafik distribusi disimpan ke: {OUT_FIG_DIST}")
print("Penentuan kriteria dan pembobotan selesai")

In [ ]:
# --- TAHAP 4 - PERHITUNGAN PROMETHEE II ------------------------------------

class PROMETHEE:
    def __init__(self, weights, criteria):
        self.weights = np.array(weights, dtype=float)
        self.criteria = list(criteria)

    def preference_function(self, d):
        return 1.0 if d > 0 else 0.0

    def compute_preference_matrix(self, matrix):
        matrix = np.asarray(matrix, dtype=float)
        n = matrix.shape[0]
        pi = np.zeros((n, n), dtype=float)
        for idx, weight in enumerate(self.weights):
            diff = matrix[:, idx, None] - matrix[None, :, idx]
            pi += weight * (diff > 0).astype(float)
        np.fill_diagonal(pi, 0.0)
        return pi

    def leaving_flow(self, pi):
        n = pi.shape[0]
        if n <= 1:
            return np.zeros(n)
        return pi.sum(axis=1) / (n - 1)

    def entering_flow(self, pi):
        n = pi.shape[0]
        if n <= 1:
            return np.zeros(n)
        return pi.sum(axis=0) / (n - 1)

    def net_flow(self, phi_plus, phi_minus):
        return phi_plus - phi_minus

    def rank(self, decision_matrix):
        result = decision_matrix.copy()
        matrix = result[self.criteria].to_numpy(dtype=float)
        pi = self.compute_preference_matrix(matrix)
        result["Phi_Plus"] = self.leaving_flow(pi)
        result["Phi_Minus"] = self.entering_flow(pi)
        result["Net_Flow"] = self.net_flow(result["Phi_Plus"].to_numpy(), result["Phi_Minus"].to_numpy())
        result = result.sort_values(["Net_Flow", "Phi_Plus"], ascending=[False, False]).reset_index(drop=True)
        result["Rank"] = np.arange(1, len(result) + 1)
        return result, pi

weights = [CRITERIA[code]["weight"] for code in criteria_codes]
promethee = PROMETHEE(weights=weights, criteria=criteria_codes)
df_promethee, pi_matrix = promethee.rank(df_clean)

print("Sudut kiri atas matriks preferensi pi (5x5):")
display(pd.DataFrame(pi_matrix[:5, :5]).round(3))

top_columns = [col for col in ["NIM", "Nama", "Phi_Plus", "Phi_Minus", "Net_Flow"] if col in df_promethee.columns]
display(df_promethee[top_columns].head(10))
print("Perhitungan PROMETHEE II selesai")

In [ ]:
# --- TAHAP 5 - RANKING HASIL ------------------------------------------------

df_result = df_promethee.sort_values(["Net_Flow", "Phi_Plus"], ascending=[False, False]).reset_index(drop=True)
df_result["Rank"] = np.arange(1, len(df_result) + 1)
df_result["Layak"] = df_result["Rank"] <= KUOTA

Path(OUT_RANKING).parent.mkdir(parents=True, exist_ok=True)
df_result.to_excel(OUT_RANKING, index=False)

print("Peringkat seluruh mahasiswa:")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(df_result)

top20 = df_result.head(20).sort_values("Net_Flow", ascending=True)
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
colors = np.where(top20["Net_Flow"] >= 0, "#2e7d32", "#c62828")
axes[0].barh(top20["Nama"], top20["Net_Flow"], color=colors)
axes[0].set_title("Net Flow 20 Mahasiswa Teratas")
axes[0].set_xlabel("Net Flow")
axes[0].set_ylabel("Mahasiswa")

scatter = axes[1].scatter(
    df_result["Phi_Plus"],
    df_result["Phi_Minus"],
    c=df_result["Net_Flow"],
    cmap="RdYlGn",
    alpha=0.75,
    edgecolor="black",
    linewidth=0.2,
)
axes[1].set_title("Leaving Flow vs Entering Flow")
axes[1].set_xlabel("Phi Plus")
axes[1].set_ylabel("Phi Minus")
fig.colorbar(scatter, ax=axes[1], label="Net Flow")
plt.tight_layout()
Path(OUT_FIG_FLOW).parent.mkdir(parents=True, exist_ok=True)
plt.savefig(OUT_FIG_FLOW, dpi=150, bbox_inches="tight")
plt.show()

print(f"Hasil ranking disimpan ke: {OUT_RANKING}")
print(f"Grafik flow disimpan ke: {OUT_FIG_FLOW}")
print("Ranking hasil selesai")

In [ ]:
# --- TAHAP 6 - VALIDASI MODEL ----------------------------------------------

label_nim_col = find_column(df_label, candidates=["NIM"], contains_any=["nim"])
if label_nim_col is None:
    raise ValueError("Kolom NIM pada data ground truth tidak ditemukan")

label_nims = set(clean_identifier(df_label[label_nim_col]).dropna())
df_eval = df_result.copy()
df_eval["y_true"] = df_eval["NIM"].isin(label_nims).astype(int)
df_eval["y_pred"] = df_eval["Layak"].astype(int)

accuracy = accuracy_score(df_eval["y_true"], df_eval["y_pred"])
precision = precision_score(df_eval["y_true"], df_eval["y_pred"], zero_division=0)
recall = recall_score(df_eval["y_true"], df_eval["y_pred"], zero_division=0)
f1 = f1_score(df_eval["y_true"], df_eval["y_pred"], zero_division=0)
cm = confusion_matrix(df_eval["y_true"], df_eval["y_pred"], labels=[0, 1])

validation_result = {
    "accuracy": float(accuracy),
    "precision": float(precision),
    "recall": float(recall),
    "f1_score": float(f1),
    "confusion_matrix": cm.astype(int).tolist(),
    "layak_implementasi": bool(accuracy >= 0.70),
}

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")

plt.figure(figsize=(6, 5))
ax = sns.heatmap(
    cm,
    annot=False,
    fmt="d",
    cmap="Greens",
    xticklabels=["Tidak Layak", "Layak"],
    yticklabels=["Tidak Ditetapkan", "Ditetapkan"],
    linewidths=0.5,
    linecolor="white",
)
threshold = cm.max() / 2
for row in range(cm.shape[0]):
    for col in range(cm.shape[1]):
        value = cm[row, col]
        text_color = "white" if value > threshold else "black"
        ax.text(col + 0.5, row + 0.5, f"{value:d}", ha="center", va="center", color=text_color, fontsize=14, fontweight="bold")
plt.title("Confusion Matrix")
plt.xlabel("Prediksi")
plt.ylabel("Aktual")
plt.tight_layout()
Path(OUT_FIG_CM).parent.mkdir(parents=True, exist_ok=True)
plt.savefig(OUT_FIG_CM, dpi=150, bbox_inches="tight")
plt.show()
print(f"Gambar confusion matrix disimpan ke: {OUT_FIG_CM}")

print("Classification Report:")
print(classification_report(df_eval["y_true"], df_eval["y_pred"], target_names=["Tidak Ditetapkan", "Ditetapkan"], zero_division=0))

if accuracy >= 0.70:
    print("Model LAYAK diimplementasikan (akurasi >= 70%)")
else:
    print("Model BELUM layak - tinjau ulang bobot atau nilai KUOTA")

print("Validasi model selesai")

In [ ]:
# --- TAHAP 7 - EXPORT MODEL DAN HASIL --------------------------------------

for folder in ["data/processed", "output/model", "output/figures"]:
    os.makedirs(folder, exist_ok=True)

joblib.dump(promethee, OUT_MODEL)

config = {
    "criteria": CRITERIA,
    "sub_criteria_map": SUB_CRITERIA_MAP,
}
with open(OUT_CONFIG, "w", encoding="utf-8") as file:
    json.dump(config, file, indent=2, ensure_ascii=False)

df_result.to_excel(OUT_RANKING, index=False)

with open(OUT_VALIDASI, "w", encoding="utf-8") as file:
    json.dump(validation_result, file, indent=2, ensure_ascii=False)

for path in [OUT_MODEL, OUT_CONFIG, OUT_RANKING, OUT_VALIDASI, CLEAN_PATH, OUT_FIG_DIST, OUT_FIG_FLOW, OUT_FIG_CM]:
    print(f"Berhasil disimpan: {Path(path).resolve()}")

print("Export model dan hasil selesai")